# RVC Inference - Google Colab (No UI)

Konversi suara dengan RVC **tanpa WebUI**, murni lewat CLI (`infer/cli.py`).

**Cara pakai:**
1. `Runtime > Change runtime type > GPU (T4)`
2. Isi parameter di Cell 1
3. Jalankan semua cell (`Runtime > Run all`) atau satu per satu
4. Upload voice model `.pth` (+ `.index`) saat diminta, atau isi URL-nya
5. Hasil audio tampil di cell terakhir


In [ ]:
#@title ## 1. Parameter
import os

REPO_URL = "https://github.com/aditiya-saputra/RVC-Inference.git"  # fork inference-only
REPO_DIR = "/content/RVC"

# ---------- Voice model ----------
MODEL_URL = ""  #@param {type:"string"}
MODEL_INDEX_URL = ""  #@param {type:"string"}
# Terima: URL langsung .pth/.index/.zip, ATAU URL halaman model
# HuggingFace (https://huggingface.co/user/repo) -> semua .pth+index diambil.

# ---------- Input audio ----------
INPUT_MODE = "upload"  #@param ["upload", "drive", "url"]
INPUT_DRIVE_PATH = "/content/drive/MyDrive/rvc-input"  #@param {type:"string"}
INPUT_URL = ""  #@param {type:"string"}

# ---------- Konversi ----------
PITCH = 0  #@param {type:"integer"}
F0_METHOD = "rmvpe"  #@param ["rmvpe", "pm"]
SPEAKER_ID = 0  #@param {type:"integer"}
INDEX_RATE = 0.75  #@param {type:"number"}
RMS_MIX_RATE = 0.25  #@param {type:"number"}
PROTECT = 0.33  #@param {type:"number"}
RESAMPLE_SR = 0  #@param {type:"integer"}
OUTPUT_FORMAT = "wav"  #@param ["wav", "flac", "mp3", "m4a"]

# ---------- Output ----------
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/rvc-output"

# ---------- Google Drive ----------
MOUNT_DRIVE = True  #@param {type:"boolean"}

WORK_DIR = "/content/rvc-work"
INPUT_DIR = WORK_DIR + "/input"
OUTPUT_DIR = WORK_DIR + "/output"
for d in (WORK_DIR, INPUT_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Parameter siap.")

In [ ]:
#@title ## 2. Mount Google Drive
from pathlib import Path

DRIVE_MOUNTED = Path("/content/drive/MyDrive").exists()
if MOUNT_DRIVE and not DRIVE_MOUNTED:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
print("Drive ter-mount:", DRIVE_MOUNTED)

In [ ]:
#@title ## 3. Clone repo + install dependensi inference
import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

!nvidia-smi -L

# torch/torchaudio sudah tersedia di Colab - jangan di-upgrade.
!pip install -q faiss-cpu librosa soundfile praat-parselmouth ffmpeg-python av transformers

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Repo siap:", REPO_DIR)

In [ ]:
#@title ## 4. Download base models (HuBERT + RMVPE)
import os
os.chdir(REPO_DIR)
!python tools/download_models.py --base

In [ ]:
#@title ## 5. Voice model (.pth / .index / .zip)
import os, zipfile, io
from pathlib import Path

os.chdir(REPO_DIR)
WEIGHTS = Path(REPO_DIR) / "assets" / "weights"
INDICES = Path(REPO_DIR) / "assets" / "indices"
WEIGHTS.mkdir(parents=True, exist_ok=True)
INDICES.mkdir(parents=True, exist_ok=True)

def install_bytes(name, data):
    lower = name.lower()
    if lower.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                install_bytes(Path(info.filename).name, zf.read(info))
    elif lower.endswith(".pth"):
        (WEIGHTS / name).write_bytes(data)
        print("model :", WEIGHTS / name)
    elif lower.endswith(".index"):
        (INDICES / name).write_bytes(data)
        print("index :", INDICES / name)
    else:
        print("skip  :", name)

if MODEL_URL.strip():
    !python tools/download_models.py --url "{MODEL_URL.strip()}"
    if MODEL_INDEX_URL.strip():
        !python tools/download_models.py --url "{MODEL_INDEX_URL.strip()}"
else:
    try:
        from google.colab import files
        print("Upload voice model (.pth / .index / .zip)...")
        for name, data in files.upload().items():
            install_bytes(name, data)
    except ImportError:
        raise RuntimeError("Bukan Colab: isi MODEL_URL / MODEL_INDEX_URL di Cell 1")

candidates = sorted(WEIGHTS.glob("*.pth"))
assert candidates, "Tidak ada file .pth di assets/weights!"
MODEL_PATH = str(candidates[0])
print("\nModel terpakai:", MODEL_PATH)

In [ ]:
#@title ## 6. Input audio
import os, shutil, subprocess
from pathlib import Path

AUDIO_EXT = {".wav", ".flac", ".mp3", ".m4a", ".ogg", ".opus", ".aac", ".wma", ".mp4", ".mkv", ".webm"}
src_dir = Path(INPUT_DIR)
src_dir.mkdir(parents=True, exist_ok=True)

def collect(folder):
    return [p for p in Path(folder).rglob("*") if p.suffix.lower() in AUDIO_EXT]

audio_files = []
if INPUT_MODE == "upload":
    from google.colab import files
    print("Upload audio...")
    for name, data in files.upload().items():
        (src_dir / name).write_bytes(data)
elif INPUT_MODE == "drive":
    assert DRIVE_MOUNTED, "Jalankan Cell 2 (Mount Google Drive) dulu atau set MOUNT_DRIVE=True"
    for p in collect(INPUT_DRIVE_PATH):
        shutil.copy2(p, src_dir / p.name)
elif INPUT_MODE == "url":
    assert INPUT_URL.strip(), "Isi INPUT_URL di Cell 1"
    name = Path(INPUT_URL.split("?")[0]).name or "input.wav"
    subprocess.run(["curl", "-L", "-o", str(src_dir / name), INPUT_URL], check=True)

audio_files = collect(src_dir)
assert audio_files, "Tidak ada audio di " + str(src_dir)
for p in audio_files:
    print("input:", p)

In [ ]:
#@title ## 7. Jalankan konversi
import os, glob, sys
os.chdir(REPO_DIR)

print("--- Speaker tersedia ---")
!python infer/cli.py --model "{MODEL_PATH}" --list-speakers

cmd = [
    sys.executable, "infer/cli.py",
    "--model", MODEL_PATH,
    "--speaker-id", str(SPEAKER_ID),
    "--input", INPUT_DIR,
    "--output", OUTPUT_DIR,
    "--pitch", str(PITCH),
    "--f0-method", F0_METHOD,
    "--index-rate", str(INDEX_RATE),
    "--rms-mix-rate", str(RMS_MIX_RATE),
    "--protect", str(PROTECT),
    "--resample-sr", str(RESAMPLE_SR),
    "--format", OUTPUT_FORMAT,
    "--overwrite",
]

# Kirim index eksplisit (pencocokan nama otomatis sering gagal jika
# nama file index mirip tapi tidak identik dengan nama model).
if INDEX_RATE > 0:
    indices = sorted(glob.glob("assets/indices/*.index"), key=os.path.getmtime)
    if indices:
        cmd += ["--index", indices[-1]]
        print("Index terpakai:", indices[-1])
    else:
        raise RuntimeError("INDEX_RATE > 0 tapi tidak ada file .index di assets/indices")

print("\n--- Konversi ---")
import subprocess
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(
        "Inference gagal (exit %s). Penyebab umum:\n"
        "1. Baris 'rvc-cli: error: ...' di atas - baca pesannya\n"
        "2. Base model belum terunduh -> jalankan ulang Cell 4\n"
        "3. Paket belum terpasang -> jalankan ulang Cell 3" % result.returncode
    )

In [ ]:
#@title ## 8. Hasil
import glob, os, shutil
from pathlib import Path
from IPython.display import Audio, display

results = sorted(glob.glob(OUTPUT_DIR + "/**/*", recursive=True))
results = [r for r in results if Path(r).is_file() and Path(r).stat().st_size > 0]
assert results, "Tidak ada output di " + OUTPUT_DIR

for r in results:
    size_mb = Path(r).stat().st_size / 1048576
    print("output: %s (%.2f MB)" % (r, size_mb))

print()
display(Audio(results[0]))

if SAVE_TO_DRIVE:
    assert DRIVE_MOUNTED, "Jalankan Cell 2 (Mount Google Drive) dulu atau set MOUNT_DRIVE=True"
    dest = Path(DRIVE_OUTPUT_DIR)
    dest.mkdir(parents=True, exist_ok=True)
    for r in results:
        shutil.copy2(r, dest / Path(r).name)
    print("Tersimpan ke:", dest)

try:
    from google.colab import files as colfiles
    for r in results:
        colfiles.download(r)
except Exception:
    pass